# Triton 内存与数据搬运 - 课后练习

本 Notebook 包含三个练习，难度递进，帮助你巩固内存管理知识。

**学习目标**：
- 掌握 2D 地址计算和 stride 处理
- 理解内存连续性对性能的影响
- 学会优化数据复用，减少重复加载
- 使用 cache hints 优化访存性能
- 处理复杂的边界情况

In [ ]:
import torch
import triton
import triton.language as tl
import time

# 检查 GPU 可用性
assert torch.cuda.is_available(), "需要 CUDA 支持的 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Triton version: {triton.__version__}")

---

## 练习 1: 2D 卷积

**目标**：实现一个高效的 2D 卷积（3×3 box filter），要求：
1. 正确处理边界和 padding
2. 支持 stride

In [ ]:
@triton.jit
def conv2d_box_filter_kernel(
    input_ptr, output_ptr,
    H, W,
    stride_h, stride_w,
    BLOCK_SIZE_H: tl.constexpr,
    BLOCK_SIZE_W: tl.constexpr,
):
    """
    TODO: 实现优化的 2D 卷积
    Y[i][j] = sum(X[i-1:i+2][j-1:j+2])
    
    步骤：
    1. 计算 Program ID
    2. 加载 (BLOCK_SIZE_H + 2) × (BLOCK_SIZE_W + 2) 的块
    3. 创建边界 mask
    4. 使用切片获取 3×3 邻居
    5. 求和
    6. 存储结果
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def conv2d_box_filter(input_tensor):
    """
    Host 端包装函数
    
    Args:
        input_tensor: (H, W) 的 tensor
    
    Returns:
        (H, W) 的输出 tensor
    """
    H, W = input_tensor.shape
    output = torch.empty_like(input_tensor)
    
    # Grid 配置
    grid = lambda meta: (
        triton.cdiv(H, meta['BLOCK_SIZE_H']),
        triton.cdiv(W, meta['BLOCK_SIZE_W']),
    )
    
    # 启动 kernel
    conv2d_box_filter_kernel[grid](
        input_tensor, output,
        H, W,
        input_tensor.stride(0), input_tensor.stride(1),
        BLOCK_SIZE_H=64,
        BLOCK_SIZE_W=64,
    )
    
    return output

In [ ]:
# 测试 2D 卷积
def test_conv2d():
    H, W = 512, 512
    input_tensor = torch.randn(H, W, device='cuda', dtype=torch.float32)
    
    # Triton 实现
    output_triton = conv2d_box_filter(input_tensor)
    
    # PyTorch 参考实现
    output_torch = torch.nn.functional.avg_pool2d(
        input_tensor.unsqueeze(0), 
        kernel_size=3, 
        stride=1, 
        padding=1
    ).squeeze(0) * 9  # avg_pool2d 会除以 9，所以乘回去
    
    # 验证
    if torch.allclose(output_triton, output_torch, atol=1e-4):
        print("✓ 2D 卷积测试通过！")
    else:
        print("✗ 2D 卷积测试失败！")
        print(f"最大误差: {torch.max(torch.abs(output_triton - output_torch)).item():.2e}")
        print(f"\n前 5x5 元素对比:")
        print(f"Triton:\n{output_triton[:5, :5]}")
        print(f"Torch:\n{output_torch[:5, :5]}")

test_conv2d()

In [ ]:
# 性能对比
def benchmark_conv2d():
    sizes = [(256, 256), (512, 512), (1024, 1024)]
    
    print(f"{'Size':>15} | {'Triton (ms)':>12} | {'Torch (ms)':>12} | {'Speedup':>10}")
    print("-" * 60)
    
    # 验证结果正确性
    for H, W in sizes:
        input_tensor = torch.randn(H, W, device='cuda', dtype=torch.float32)
        output = conv2d_box_filter(input_tensor)
        assert torch.allclose(output, torch.nn.functional.avg_pool2d(
            input_tensor.unsqueeze(0), kernel_size=3, stride=1, padding=1
        ).squeeze(0) * 9, atol=1e-6)

    torch.cuda.synchronize()

    # 性能测试
    for H, W in sizes:
        input_tensor = torch.randn(H, W, device='cuda', dtype=torch.float32)
        
        # Triton 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output = conv2d_box_filter(input_tensor)
        torch.cuda.synchronize()
        t_triton = (time.time() - t0) * 1000
        
        # PyTorch 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output_torch = torch.nn.functional.avg_pool2d(
                input_tensor.unsqueeze(0), kernel_size=3, stride=1, padding=1
            ).squeeze(0) * 9
        torch.cuda.synchronize()
        t_torch = (time.time() - t0) * 1000
        
        speedup = t_torch / t_triton
        print(f"{H}x{W:>10} | {t_triton:>12.2f} | {t_torch:>12.2f} | {speedup:>10.2f}x")

benchmark_conv2d()

## 练习 2: 优化矩阵乘法内存访问

**目标**：实现一个高效的矩阵乘法 kernel，运用 cache hints 优化访存

**提示**：
1. 使用 `cache_modifier` 参数优化加载
2. 注意 stride 的正确传递
3. 处理非 BLOCK_SIZE 整数倍的情况
4. 使用 Block Ptr 应该怎么写

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    """
    TODO: 实现带 cache hints 的矩阵乘法
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass


# 把上面的代码复制下来，去掉 cache hints 参数
@triton.jit
def matmul_kernel_no_cache(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pass


def matmul(a, b, use_cache_hints=True):
    """
    Host 端包装函数

    Args:
        a: (M, K) tensor
        b: (K, N) tensor
        use_cache_hints: 是否使用cache hints

    Returns:
        (M, N) tensor
    """
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "矩阵维度不匹配"

    c = torch.empty(M, N, device=a.device, dtype=a.dtype)

    # Grid 配置
    grid = lambda meta: (
        triton.cdiv(M, meta['BLOCK_SIZE_M']),
        triton.cdiv(N, meta['BLOCK_SIZE_N']),
    )

    # 根据参数选择kernel
    kernel = matmul_kernel if use_cache_hints else matmul_kernel_no_cache

    # 启动 kernel
    kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=64,
        BLOCK_SIZE_N=64,
        BLOCK_SIZE_K=32,
    )

    return c

In [ ]:
# 测试矩阵乘法
def test_matmul():
    M, N, K = 512, 512, 512

    # 使用较小的随机数范围来减少数值误差
    a = torch.randn(M, K, device='cuda', dtype=torch.float32) * 0.1
    b = torch.randn(K, N, device='cuda', dtype=torch.float32) * 0.1
    
    # Triton 实现
    c_triton = matmul(a, b)
    
    # PyTorch 参考实现
    c_torch = torch.matmul(a, b)

    # 验证
    if torch.allclose(c_triton, c_torch, atol=1e-3):
        print("✓ 矩阵乘法测试通过！")
    else:
        print("✗ 矩阵乘法测试失败！")
        print(f"最大误差: {torch.max(torch.abs(c_triton - c_torch)).item():.2e}")

test_matmul()

In [ ]:
def benchmark_cache_hints():
    """
    对比使用和不使用cache hints的性能差异
    """
    print("\n" + "="*60)
    print("Cache Hints 性能对比测试")
    print("="*60)

    # 测试不同的矩阵大小
    sizes = [(2048, 2048, 2048), (4096, 4096, 4096), (8192, 8192, 8192)]

    for M, N, K in sizes:
        print(f"\n矩阵大小: {M}x{K} @ {K}x{N} = {M}x{N}")
        print("-" * 40)

        # 创建测试数据
        a = torch.randn(M, K, device='cuda', dtype=torch.float32) * 0.1
        b = torch.randn(K, N, device='cuda', dtype=torch.float32) * 0.1

        # 预热
        for _ in range(3):
            _ = matmul(a, b, use_cache_hints=True)
            _ = matmul(a, b, use_cache_hints=False)
        torch.cuda.synchronize()

        # 测试带cache hints的版本
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(10):
            c_with_cache = matmul(a, b, use_cache_hints=True)
        torch.cuda.synchronize()
        time_with_cache = (time.time() - start_time) / 10

        # 测试不带cache hints的版本
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(10):
            c_no_cache = matmul(a, b, use_cache_hints=False)
        torch.cuda.synchronize()
        time_no_cache = (time.time() - start_time) / 10

        # 验证结果一致性
        max_diff = torch.max(torch.abs(c_with_cache - c_no_cache)).item()

        # 计算FLOPS
        flops = 2 * M * N * K  # 矩阵乘法的浮点运算数
        flops_with_cache = flops / time_with_cache / 1e12  # TFLOPS
        flops_no_cache = flops / time_no_cache / 1e12  # TFLOPS

        # 输出结果
        print(f"带Cache Hints:    {time_with_cache*1000:.2f} ms ({flops_with_cache:.2f} TFLOPS)")
        print(f"不带Cache Hints:  {time_no_cache*1000:.2f} ms ({flops_no_cache:.2f} TFLOPS)")
        print(f"性能提升:         {time_no_cache/time_with_cache:.2f}x")
        print(f"结果误差:         {max_diff:.2e}")

        # PyTorch参考性能
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(10):
            c_torch = torch.matmul(a, b)
        torch.cuda.synchronize()
        time_torch = (time.time() - start_time) / 10
        flops_torch = flops / time_torch / 1e12

        print(f"PyTorch参考:      {time_torch*1000:.2f} ms ({flops_torch:.2f} TFLOPS)")
        print(f"vs PyTorch:       {time_torch/time_with_cache:.2f}x (带cache hints)")

NVIDIA 现代GPU具有非常智能的 L1/L2 缓存管理，GPU可能已经自动优化了内存访问模式。

虽然在这个测试中效果不明显，但cache hints仍有价值：

1. 代码意图表达：明确告诉编译器你的访问模式
2. 编译器提示：帮助生成更优化的代码


## 总结

完成这三个练习后，你应该掌握了：
- 2D/多维地址计算和 mask 处理
- Cache hints 的实际应用

**下一步**：学习 Triton 的 Reduction 与原子操作！

---

## 课后答案

### 练习 1：2D 卷积

```python
@triton.jit
def conv2d_box_filter_kernel(
    input_ptr, output_ptr,
    H, W,
    stride_h, stride_w,
    BLOCK_SIZE_H: tl.constexpr,
    BLOCK_SIZE_W: tl.constexpr,
):
    pid_h = tl.program_id(0)
    pid_w = tl.program_id(1)
    
    # 生成当前块内每个线程对应的 h 和 w 的偏移量
    # 使用 broadcasting技巧 ([:, None] 和 [None, :]) 生成 2D 网格
    offs_h = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H)
    offs_w = pid_w * BLOCK_SIZE_W + tl.arange(0, BLOCK_SIZE_W)
    
    r_idx = offs_h[:, None]
    c_idx = offs_w[None, :]
    mask = (r_idx < H) & (c_idx < W)

    acc = tl.zeros([BLOCK_SIZE_H, BLOCK_SIZE_W], dtype=tl.float32)
    
    # 循环加载 3x3 邻居并求和
    for dy in range(-1, 2):
        for dx in range(-1, 2):
            neighbor_h = r_idx + dy
            neighbor_w = c_idx + dx
            
            mask_in_h = (neighbor_h >= 0) & (neighbor_h < H)
            mask_in_w = (neighbor_w >= 0) & (neighbor_w < W)
            mask_in = mask_in_h & mask_in_w

            # 利用广播技巧计算每个邻居的指针偏移量
            offset = neighbor_h * stride_h + neighbor_w * stride_w
            input_ptrs = input_ptr + offset
            val = tl.load(input_ptrs, mask=mask_in, other=0.0)
            acc += val

    # 存储结果
    output_ptrs = output_ptr + r_idx * stride_h + c_idx * stride_w
    tl.store(output_ptrs, acc, mask=mask)

### 练习 2：矩阵乘法

不使用 block ptr 的版本：

```python
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    rm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    rn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)

    acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    for k in range(0, K, BLOCK_SIZE_K):
        rk = k + tl.arange(0, BLOCK_SIZE_K)

        # 创建 mask
        mask_a = (rm[:, None] < M) & (rk[None, :] < K)  # (64, 32)
        mask_b = (rk[:, None] < K) & (rn[None, :] < N)  # (32, 64)

        # 计算指针
        a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
        b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn

        a = tl.load(a_ptrs, mask=mask_a, other=0.0, cache_modifier=".cg")
        b = tl.load(b_ptrs, mask=mask_b, other=0.0, cache_modifier=".cg")

        acc += tl.dot(a, b)

    cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)

    c_ptrs = c_ptr + cm[:, None] * stride_cm + cn[None, :] * stride_cn
    out_mask = (cm[:, None] < M) & (cn[None, :] < N)
    tl.store(c_ptrs, acc, mask=out_mask, cache_modifier=".cg")
```

使用 block ptr 的版本：

```python
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    """
    使用 block pointer 的矩阵乘法 kernel
    Block pointer 提供了更简洁的内存访问模式和更好的性能
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # 计算当前block的起始位置
    offs_m = pid_m * BLOCK_SIZE_M
    offs_n = pid_n * BLOCK_SIZE_N

    # 创建 A 矩阵的 block pointer
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr,
        shape=(M, K),
        strides=(stride_am, stride_ak),
        offsets=(offs_m, 0),
        block_shape=(BLOCK_SIZE_M, BLOCK_SIZE_K),
        order=(1, 0)
    )

    # 创建 B 矩阵的 block pointer
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr,
        shape=(K, N),
        strides=(stride_bk, stride_bn),
        offsets=(0, offs_n),
        block_shape=(BLOCK_SIZE_K, BLOCK_SIZE_N),
        order=(1, 0)
    )

    # 累加器初始化
    acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    # K 维度循环
    for k in range(0, K, BLOCK_SIZE_K):
        # 使用 block pointer 加载数据，自动处理边界检查
        a = tl.load(a_block_ptr, boundary_check=(0, 1), padding_option="zero")
        b = tl.load(b_block_ptr, boundary_check=(0, 1), padding_option="zero")

        # 矩阵乘法累加
        acc += tl.dot(a, b)

        # 更新 block pointer 的 K 维度偏移
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_SIZE_K)) # 按列移动
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_SIZE_K, 0)) # 按行移动

    # 创建输出 C 矩阵的 block pointer
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr,
        shape=(M, N),
        strides=(stride_cm, stride_cn),
        offsets=(offs_m, offs_n),
        block_shape=(BLOCK_SIZE_M, BLOCK_SIZE_N),
        order=(1, 0)
    )

    # 存储结果，自动处理边界检查
    tl.store(c_block_ptr, acc, boundary_check=(0, 1))
```
